## License
Copyright 2026 jphall@gwu.edu. MIT License; see the repository LICENSE file.

# Mini multiple-choice benchmark

This MMLU-like mini benchmark asks fixed multiple-choice questions, normalizes the reply, and calculates accuracy. It is intentionally small so students can inspect every step; realistic benchmarking needs many more questions and repeated runs.

Install the root requirements file, set GW_AZURE_OPENAI_KEY, and use the project VS Code kernel.


## 1. Import packages and configure Azure


In [1]:
# 1. Import packages and configure Azure
# Load the libraries and create one Azure Chat Completions client.

import os
from getpass import getpass

import pandas as pd
from openai import AzureOpenAI

RESOURCE = "gw-sb-01"
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")
client = AzureOpenAI(
    azure_endpoint=ENDPOINT,
    api_key=api_key,
    api_version="2024-12-01-preview",
)


## 2. Define questions and ask the model


In [2]:
# 2. Define questions and ask the model
# Keep the benchmark items and answer-normalization logic visible.

questions = [
    {
        "question": "What is the capital of France?",
        "choices": ["A. Berlin", "B. Madrid", "C. Paris", "D. Rome"],
        "answer": "C",
    },
    {
        "question": "Which data structure uses first-in, first-out order?",
        "choices": ["A. Stack", "B. Queue", "C. Tree", "D. Graph"],
        "answer": "B",
    },
]

def ask_model(item):
    prompt = item["question"] + "\n" + "\n".join(item["choices"])
    prompt += "\nReply only A, B, C, or D."

    # GPT-5 uses some tokens for internal reasoning; 500 leaves room for a visible answer.
    reply = client.chat.completions.create(
        model="gpt-5-nano",
        messages=[{"role": "user", "content": prompt}],
        max_completion_tokens=500,
    )

    # An empty value is possible if a service response has no visible text.
    answer = reply.choices[0].message.content or ""
    return answer.strip().upper()[:1]

results = [
    {
        "question": item["question"],
        "expected": item["answer"],
        "prediction": ask_model(item),
    }
    for item in questions
]

for row in results:
    row["correct"] = row["expected"] == row["prediction"]

print(results)


[{'question': 'What is the capital of France?', 'expected': 'C', 'prediction': 'C', 'correct': True}, {'question': 'Which data structure uses first-in, first-out order?', 'expected': 'B', 'prediction': 'B', 'correct': True}]


## 3. Score the mini benchmark


In [3]:
# 3. Score the mini benchmark
# Convert results to a table and calculate simple accuracy.

results = pd.DataFrame(results)

print(f"Accuracy: {results.correct.mean():.0%}")

results


Accuracy: 100%


,question,expected,prediction,correct
0,What is the capital of France?,C,C,True
1,"Which data structure uses first-in, first-out ...",B,B,True
